<div style="text-align: justify; max-width: 90ch">

# Lecture 03f: DSPy prompt optimization

**Status: Optional / career-track.**

`lec_03b` compared two variants that a person chose. An **optimizer** chooses instead: it tries prompts on questions you give it and keeps what scores best,
as `GridSearchCV` (`lec_13a` of the Python course) tries hyperparameters. [DSPy](https://dspy.ai/) is such a framework; `lec_01d` left it out because it writes the prompt for you.
Here it writes the prompt of lecture 2's RAG system, and the scorers of `lec_03a` say whether it beats the hand-written prompt on questions it never saw.

Needs the optional group `uv sync --group career-track`: guide [`03a_dspy_optional_group`](../infra_tools/03a_dspy_optional_group.md).

**What this notebook covers**

- A DSPy **signature** and **module** on the local model.
- Dev and test questions, split by topic, and a **metric**.
- `BootstrapFewShot`: the optimizer picks worked examples for the prompt.
- Three prompts compared on the test questions, and what the metric left out.

</div>

In [1]:
import os
from pathlib import Path

import dspy
import mlflow
import pandas as pd
from dotenv import load_dotenv
from langchain_ollama import ChatOllama
from mlflow.genai.scorers import scorer

from course_helpers.agents import total_tokens
from course_helpers.checks import check_mlflow
from course_helpers.corpus import load_corpus, load_eval_set
from course_helpers.evaluation import (
    DETERMINISTIC_SCORERS,
    compare_variants,
    fact_keywords_found,
    to_eval_records,
)
from course_helpers.rag import (
    LECTURE_KINDS,
    build_chroma_index,
    format_context,
    make_rag_chain,
    split_documents,
)

In [2]:
# --- Course configuration: edit TIER only; settings in .env (Read: 01d_env_hugging_face) ---
TIER = "cpu"  # 8-16 GB RAM, no GPU (default)
# TIER = "gpu"  # >= 8 GB VRAM: bigger, better answers (pull qwen3:8b first)

REPO = Path.cwd().resolve().parents[1]  # runs from a lecture subfolder
load_dotenv(REPO / ".env")  # load your .env settings: OLLAMA_HOST, HF_TOKEN, ...
TIER = os.environ.get("COURSE_TIER", TIER)  # tools/run_all.py can override
from huggingface_hub.utils import disable_progress_bars

disable_progress_bars()  # no download bars in saved outputs (works after the imports)
CHAT_MODEL = {"cpu": "qwen3:1.7b", "gpu": "qwen3:8b"}[TIER]
EMBED_MODEL = "nomic-embed-text"
OLLAMA_HOST = os.environ.get("OLLAMA_HOST", "http://localhost:11434")
MLFLOW_URI = os.environ.get("MLFLOW_TRACKING_URI", "http://127.0.0.1:5010")
# To override .env in this notebook only, uncomment and edit (never a token):
# OLLAMA_HOST = "http://192.168.1.20:11434"
# MLFLOW_URI = "http://127.0.0.1:5011"
EXPERIMENT = "llm-course-03-eval"
DATA_DIR = REPO / "data"  # gitignored runtime state (indexes, caches)

from course_helpers.checks import check_ollama

check_ollama(OLLAMA_HOST, [CHAT_MODEL])

Ollama OK at http://localhost:11434; models ready: qwen3:1.7b


<div style="text-align: justify; max-width: 90ch">

**Rebuild inputs.** The system of `lec_03a`, from the `course_helpers` helpers:

- the Chroma index of `lec_02d`, reused if it exists (otherwise built now, which takes minutes);
- four chunks per question, from the lecture notes and guides (the filter of `lec_02d`);
- the chat model with thinking off, temperature 0 and at most 200 answer tokens, and the hand-written prompt of `lec_02d` (`make_rag_chain`);
- the 19 questions of the evaluation set.

</div>

In [3]:
check_ollama(OLLAMA_HOST, [EMBED_MODEL])

chunks = split_documents(load_corpus(REPO / "corpus" / "uoa_py_course"))
store = build_chroma_index(
    chunks,
    "course_notes",
    DATA_DIR / "chroma",
    embed_model=EMBED_MODEL,
    ollama_host=OLLAMA_HOST,
)
retriever = store.as_retriever(
    search_kwargs={"k": 4, "filter": {"kind": {"$in": LECTURE_KINDS}}}
)

MAX_ANSWER_TOKENS = 200
llm = ChatOllama(
    model=CHAT_MODEL,
    base_url=OLLAMA_HOST,
    reasoning=False,
    temperature=0,
    num_predict=MAX_ANSWER_TOKENS,
    num_ctx=8192,
)
rag_chain = make_rag_chain(retriever, llm)

eval_set = load_eval_set(REPO / "corpus" / "eval" / "qa_eval_set.jsonl")

print(len(eval_set), "questions in the evaluation set")

Ollama OK at http://localhost:11434; models ready: nomic-embed-text


Reusing 'course_notes': 2452 chunks in chroma/
19 questions in the evaluation set


<div style="text-align: justify; max-width: 90ch">

## 1. A prompt is a setting chosen by hand

The prompt of `lec_02d` is three sentences someone wrote and never tested against another wording. In the Python course a search (`lec_13a`) replaced hyperparameters chosen by hand.
DSPy does the same for prompts:

| Term | Meaning |
| --- | --- |
| **signature** | what goes in and what comes out, e.g. `context, question -> answer`, plus instructions |
| **module** | a step that calls the model with a signature; `dspy.Predict` is the simplest |
| **demo** | a worked example (inputs and a good output) put into the prompt |
| **metric** | a function that says whether one output is good |
| **optimizer** | tries prompts or demos on the **dev** questions and keeps what the metric likes |
| **compile** | DSPy's word for running an optimizer; it returns the optimized module |

</div>

<div style="text-align: justify; max-width: 90ch">

## 2. DSPy on the local model

`dspy.LM` reaches models through [LiteLLM](https://docs.litellm.ai/docs/providers/ollama): `"ollama_chat/<model>"` means Ollama's chat endpoint at `api_base`.
The settings are those of `llm` above, in DSPy's names (`max_tokens` for `num_predict`, `think=False` for `reasoning=False`),
plus one: **`cache=False`**. DSPy otherwise stores every answer on disk and replays it, so a rerun would never ask the model again.
`dspy.configure` makes it the model of every DSPy module in this notebook.

</div>

In [4]:
lm = dspy.LM(
    f"ollama_chat/{CHAT_MODEL}",
    api_base=OLLAMA_HOST,
    temperature=0,
    max_tokens=MAX_ANSWER_TOKENS,
    num_ctx=8192,
    think=False,
    cache=False,
)
dspy.configure(lm=lm)

<div style="text-align: justify; max-width: 90ch">

[MLflow's DSPy autolog](https://mlflow.org/docs/latest/genai/flavors/dspy/) traces the DSPy calls of this notebook, but not those made during an optimization,
unless you also pass `log_traces_from_compile=True`. `log_compiles=True` records each optimization as a run instead (section 5).

</div>

In [5]:
check_mlflow(MLFLOW_URI)
mlflow.set_tracking_uri(MLFLOW_URI)
mlflow.set_experiment(EXPERIMENT)
mlflow.tracing.disable_notebook_display()  # keep the trace viewer out of the saved notebook
mlflow.langchain.autolog()
mlflow.dspy.autolog(log_compiles=True)

MLflow OK at http://127.0.0.1:5010


<div style="text-align: justify; max-width: 90ch">

The signature takes the instructions of the `lec_02d` prompt. DSPy's documentation usually writes a signature as a class;
the string form below does the same without one. `dspy.Predict` turns the signature into a module, called with keyword arguments.
The retriever stays outside DSPy: it fetches the chunks, and DSPy only writes the prompt around them.

</div>

In [6]:
INSTRUCTIONS = (
    "You answer questions about a Python course using only the notes in the context. "
    "If the notes do not contain the answer, say so. Mention the source file you used."
)
signature = dspy.Signature("context, question -> answer", INSTRUCTIONS)
qa = dspy.Predict(signature)

q19 = next(q for q in eval_set if q["id"] == "q19")
prediction = qa(
    context=format_context(retriever.invoke(q19["question"])), question=q19["question"]
)

print(prediction.answer)

The difference between a list and a tuple is that a list is mutable (can be modified after creation) while a tuple is immutable (cannot be modified after creation). 

The source file used: lecture_04/lec_04b_index_slice_list_tuple_set.md


<div style="text-align: justify; max-width: 90ch">

A module returns a **Prediction**, whose fields are the outputs of the signature. Check this answer against the evaluation set:
does it say that lists are mutable and tuples are not, and does it name `lec_04b`?

Nobody wrote the prompt that produced it. `lm.history` keeps every call; the first message of the last one is the system prompt DSPy built from the signature:

</div>

In [7]:
print(lm.history[-1]["messages"][0]["content"])

Your input fields are:
1. `context` (str): 
2. `question` (str):
Your output fields are:
1. `answer` (str):
All interactions will be structured in the following way, with the appropriate values filled in.

[[ ## context ## ]]
{context}

[[ ## question ## ]]
{question}

[[ ## answer ## ]]
{answer}

[[ ## completed ## ]]
In adhering to this structure, your objective is: 
        You answer questions about a Python course using only the notes in the context. If the notes do not contain the answer, say so. Mention the source file you used.


<div style="text-align: justify; max-width: 90ch">

The instructions sit at the end, after a description of the fields and of the reply format: the model must answer between markers such as `[[ ## answer ## ]]`, which DSPy parses back into `prediction.answer`.
This is what `lec_01d` meant: the prompt is no longer in your hands.

</div>

<div style="text-align: justify; max-width: 90ch">

## 3. Dev and test questions, split by topic

`lec_03a` graded on the same questions that chose `k=4`, and said so. An optimizer fits its questions much harder, so here they are split, and the test half is used once, at the end:

- **dev** (q02 to q06): lecture 13, `GridSearchCV` and MLflow;
- **test** (q08, q10, q12, q16, q19): lectures 4 to 12, models and Python basics.

**The halves differ in kind**, on purpose. Demos about MLflow cannot help with the content of a pandas question, only with the form of the answer:
a gain on test is a gain in form that carries over to other topics. With five questions a side, every comparison is coarse.

The optimizer reads DSPy **examples**: `with_inputs` names the fields the module receives; the expectations are for the metric only.

</div>

In [8]:
DEV_IDS = ["q02", "q03", "q04", "q05", "q06"]
TEST_IDS = ["q08", "q10", "q12", "q16", "q19"]
dev_set = [q for q in eval_set if q["id"] in DEV_IDS]
test_set = [q for q in eval_set if q["id"] in TEST_IDS]

dev_examples = [
    dspy.Example(
        context=format_context(retriever.invoke(record["inputs"]["question"])),
        question=record["inputs"]["question"],
        expectations=record["expectations"],
    ).with_inputs("context", "question")
    for record in to_eval_records(dev_set)
]

print(
    len(dev_examples),
    "dev examples; the module receives",
    list(dev_examples[0].inputs()),
)

5 dev examples; the module receives ['context', 'question']


<div style="text-align: justify; max-width: 90ch">

## 4. The metric

An optimizer needs a verdict per answer, computed many times: here a deterministic scorer, cheap and repeatable; a judge also works, at one more model call per answer, many times over.
The metric is the keyword check of `lec_03a`: an answer **passes** when `fact_keywords_found` is at least 0.5. It does not look at the source file.

A DSPy metric receives the example and the prediction (and a `trace`, unused here); an MLflow scorer receives `outputs` and `expectations`.
Both call the same check. Test it on two answers to q02 written by hand:

</div>

In [9]:
def dspy_metric(example, prediction, trace=None):
    """The optimizer's metric: at least half of the expected facts found."""
    found = fact_keywords_found(
        outputs=prediction.answer, expectations=example.expectations
    )
    return found >= 0.5


@scorer
def metric_passed(outputs, expectations):
    """The same check as an MLflow scorer, for the evaluation runs."""
    return fact_keywords_found(outputs=outputs, expectations=expectations) >= 0.5


q02_example = dev_examples[0]
HAND_ANSWERS = {
    "facts, no source": "GridSearchCV runs cross-validation for every combination "
    "in the hyperparameter grid and keeps the best one in best_params_.",
    "source, no facts": "See lec_13a_gridsearchcv_hyperparameter_tuning.",
}

pd.Series(
    {
        name: dspy_metric(q02_example, dspy.Prediction(answer=text))
        for name, text in HAND_ANSWERS.items()
    }
)

facts, no source     True
source, no facts    False
dtype: bool

<div style="text-align: justify; max-width: 90ch">

The answer with the facts passes without naming any file; the one that names the right file fails. That holds on every run:
the metric reads text, not a model. Keep it in mind for section 6.

</div>

<div style="text-align: justify; max-width: 90ch">

## 5. The optimizer: `BootstrapFewShot`

[`BootstrapFewShot`](https://dspy.ai/api/optimizers/BootstrapFewShot/) keeps the instructions and adds demos:

1. it runs the module on the dev questions, one at a time;
2. it keeps each answer the metric passes, with its inputs, as a demo;
3. it stops at `max_bootstrapped_demos`, and puts the demos into every later prompt.

`max_labeled_demos=0`: the evaluation set has expected facts, not written answers, so there is nothing else to copy. Each dev question it tries costs one model call.
In the named run, `log_compiles` records the model settings (`lm_params`), the dev set (`trainset.json`) and the optimized module with its demos (`best_model.json`).

</div>

In [10]:
optimizer = dspy.BootstrapFewShot(
    metric=dspy_metric, max_bootstrapped_demos=2, max_labeled_demos=0
)

with mlflow.start_run(run_name="BootstrapFewShot"):
    optimized_qa = optimizer.compile(qa, trainset=dev_examples)

print(len(optimized_qa.demos), "demos")

Bootstrapped 2 full traces after 2 examples for up to 1 rounds, amounting to 2 attempts.
🏃 View run BootstrapFewShot at: http://127.0.0.1:5010/#/experiments/4/runs/1105b81c0bf446d0aa498206337e24c9
🧪 View experiment at: http://127.0.0.1:5010/#/experiments/4
2 demos


<div style="text-align: justify; max-width: 90ch">

Which dev questions became demos, and does each demo answer name a source file? A file name contains `lec_`:

</div>

In [11]:
id_of = {q["question"]: q["id"] for q in eval_set}
for demo in optimized_qa.demos:
    print(id_of[demo.question], "| answer names a file:", "lec_" in demo.answer)

q02 | answer names a file: False
q03 | answer names a file: False


<div style="text-align: justify; max-width: 90ch">

Each demo is a dev question with the model's own answer, which passed the metric. If no answer passed, there would be no demos and the optimized module would be the one you started with:
**the optimizer can only pick from what the model already does well.** Other DSPy optimizers search further, at many more calls:
[`MIPROv2`](https://dspy.ai/api/optimizers/MIPROv2/) also has the model propose new instructions, and [`GEPA`](https://dspy.ai/api/optimizers/GEPA/overview/) rewrites them after reading failed answers.

</div>

<div style="text-align: justify; max-width: 90ch">

## 6. Three prompts on the test questions

The method of `lec_03b`: same questions, same scorers, one named evaluation run each.

- **hand-written**: the `lec_02d` chain;
- **DSPy, no demos**: the signature alone, with DSPy's prompt;
- **DSPy, optimized**: the same, with the demos.

Each DSPy variant retrieves inside its predict function, decorated with `@mlflow.trace` (`lec_02b`): the search and the DSPy call then land in one trace,
where `retrieval_hit` finds the `RETRIEVER` span (without the decorator it reads 0). The scorers: the five of `lec_03a`,
`metric_passed`, and `total_tokens` (`course_helpers.agents`), every token read and written for one answer, because every demo is read again with every question.

</div>

In [12]:
def predict_hand_written(question):
    """The lec_02d chain: the prompt a person wrote."""
    return rag_chain.invoke(question)


@mlflow.trace
def predict_dspy(question):
    """DSPy's prompt from the signature alone."""
    context = format_context(retriever.invoke(question))
    return qa(context=context, question=question).answer


@mlflow.trace
def predict_dspy_optimized(question):
    """DSPy's prompt with the demos the optimizer picked."""
    context = format_context(retriever.invoke(question))
    return optimized_qa(context=context, question=question).answer


SCORERS = [*DETERMINISTIC_SCORERS, metric_passed, total_tokens]
test_records = to_eval_records(test_set)
results_by_variant = {}

<div style="text-align: justify; max-width: 90ch">

One cell per variant, five answers each.

</div>

In [13]:
with mlflow.start_run(run_name="hand-written"):
    results_by_variant["hand-written"] = mlflow.genai.evaluate(
        data=test_records, predict_fn=predict_hand_written, scorers=SCORERS
    )

2026/10/11 17:14:12 INFO mlflow.models.evaluation.utils.trace: Auto tracing is temporarily enabled during the model evaluation for computing some metrics and debugging. To disable tracing, call `mlflow.autolog(disable=True)`.


2026/10/11 17:14:12 INFO mlflow.genai.utils.data_validation: Testing model prediction with the first sample in the dataset. To disable this check, set the MLFLOW_GENAI_EVAL_SKIP_TRACE_VALIDATION environment variable to True.


In [14]:
with mlflow.start_run(run_name="DSPy, no demos"):
    results_by_variant["DSPy, no demos"] = mlflow.genai.evaluate(
        data=test_records, predict_fn=predict_dspy, scorers=SCORERS
    )

2026/10/11 17:14:20 INFO mlflow.genai.utils.data_validation: Testing model prediction with the first sample in the dataset. To disable this check, set the MLFLOW_GENAI_EVAL_SKIP_TRACE_VALIDATION environment variable to True.


In [15]:
with mlflow.start_run(run_name="DSPy, optimized"):
    results_by_variant["DSPy, optimized"] = mlflow.genai.evaluate(
        data=test_records, predict_fn=predict_dspy_optimized, scorers=SCORERS
    )

2026/10/11 17:14:26 INFO mlflow.genai.utils.data_validation: Testing model prediction with the first sample in the dataset. To disable this check, set the MLFLOW_GENAI_EVAL_SKIP_TRACE_VALIDATION environment variable to True.


In [16]:
compare_variants(results_by_variant).round(2)

,hand-written,"DSPy, no demos","DSPy, optimized"
answer_tokens/max,200.00,149.00,146.00
answer_tokens/mean,166.20,107.80,111.00
cites_expected_source/mean,0.60,0.20,0.00
fact_keywords_found/mean,0.83,0.93,0.90
metric_passed/mean,1.00,1.00,1.00
refused/mean,0.00,0.00,0.00
retrieval_hit/mean,1.00,1.00,1.00
seconds_per_answer/mean,3.96,2.35,2.71
total_tokens/mean,866.80,947.40,2452.60


<div style="text-align: justify; max-width: 90ch">

Read the table one row at a time:

- **`metric_passed/mean`** is what the optimizer was told to raise. In example runs every variant scored 1.0, or one answer short of it:
  on these questions there was little left to gain.
- **`total_tokens/mean`**: every optimized prompt carries the demos, each with its own four chunks, so if the optimizer found demos this column reads several times the others (about two to three times in example runs).
  In example runs the input of one answer grew from about 840 tokens without demos to about 2,350 with them.
- **`cites_expected_source/mean`**: all three prompts ask for the source file. In example runs the optimized module named it in no answer,
  against about one answer in five without demos and two or three in five for the hand-written prompt.

With five questions, each step of a mean is one answer.

</div>

<div style="text-align: justify; max-width: 90ch">

## 7. You get what you measure

The optimizer did what it was asked: it kept answers that state the facts. The source line was never part of the metric:
if no demo answer printed in section 5 names a file, the optimized module has no example of a source line. Compare `cites_expected_source` in the table above.
**You get what you measure**: the scorers chosen in `lec_03a` are the real specification of the system, and an optimizer follows them more literally than any person.

Before you optimize:

- **Put every requirement in the metric**, here also `cites_expected_source`.
- **Count the cost**: demos are paid in tokens on every call.
- **Grade on held-out questions** with all the scorers, not only the metric.

</div>

<div style="text-align: justify; max-width: 90ch">

### ⏸ Pause point

In the MLflow UI, open the run `BootstrapFewShot` and its artifact `best_model.json`: are its demos the ones printed in section 5, and is each answer right?
Which metric would you write so that the optimizer keeps the source line?

</div>

<div style="text-align: justify; max-width: 90ch">

## Recap

- A prompt is a setting chosen by hand; an **optimizer** tries alternatives on **dev** questions and keeps what a **metric** likes.
- DSPy builds the prompt from a **signature** (`"context, question -> answer"` plus instructions); `dspy.Predict` is the simplest **module**.
  The prompt is no longer in your hands.
- Here the metric is a deterministic scorer from `lec_03a`: cheap and repeatable. A judge also works, at one more model call per answer, many times over.
- **`BootstrapFewShot`** keeps the model's own passing dev answers as **demos**; every demo costs tokens on every call.
- Grade once, on **held-out** questions, with every scorer you care about: **the optimizer raises what the metric measures**, and nothing else.

**Next:** the lecture 3 exercises, and the [final assignment](../../final_assignment/README.md).

</div>